# Lesson 1.3 · Assumptions and diagnostics

The coefficients of a linear regression have a meaning only if the model is reasonable for that data. This
lesson gives you the tools to check it: residual plots, the multicollinearity index (VIF) and the measures
of leverage and influence.

**What you will learn**

- Which assumptions matter for predicting and which for interpreting.
- To read residual plots and to recognize nonlinearity and heteroscedasticity.
- To compute VIF, leverage and Cook's distance with a few lines of numpy.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.linear_model import LinearRegression
from common.plots import residual_panel

## The assumptions, and what they break

| Assumption | If it is violated | It compromises |
|---|---|---|
| Linearity of the relationship | systematic error | predictions and interpretation |
| Independence of the errors | wrong standard errors | mostly inference |
| Constant variance (homoscedasticity) | wrong standard errors | mostly inference |
| Normal residuals | inaccurate t and F tests on small samples | inference in small samples |
| No strong multicollinearity | unstable coefficients | interpretation, not predictions |

If you only care about predicting, linearity is the assumption that really matters. If you want to explain the effect of a
variable, they all matter.

## Looking at the residuals

The most informative plot is the one of the residuals against the predicted values. With an adequate model it must look like a
cloud with no structure, centered on zero and of constant height. Any recognizable shape is a signal.

In [ ]:
rng = np.random.default_rng(1)
n = 300
x = rng.uniform(1, 10, n)
cases = {
    "assumptions respected": 2 + 1.5 * x + rng.normal(0, 1.5, n),
    "curved relationship":   2 + 0.6 * (x - 5) ** 2 + rng.normal(0, 1.5, n),
    "growing variance":      2 + 1.5 * x + rng.normal(0, 1, n) * 0.4 * x,
}
for name, y in cases.items():
    m = LinearRegression().fit(x.reshape(-1, 1), y)
    residual_panel(y, m.predict(x.reshape(-1, 1)), name)

In the first case the residuals are a horizontal cloud and the Q-Q plot follows the diagonal. In the second the residuals draw
a **U**: the line errs low in the middle and high at the sides. In the third they form a **funnel**: the error
grows with the predicted value. The two pathologies have different remedies (transforming the features or the target in the first
case, robust estimators or robust standard errors in the second).

## Multicollinearity: the VIF

The **Variance Inflation Factor** of a feature answers one question: how well can the other features
reconstruct it? It is computed as 1 / (1 − R²), where R² is the one from the regression of that feature on all the others.
A VIF of 10 means that the variance of its coefficient is inflated tenfold compared to the case of
uncorrelated features. The usual alarm thresholds are 5 and 10.

In [ ]:
def vif(X):
    """VIF of every column: 1 / (1 - R2) of the regression of that column on the others."""
    X = pd.DataFrame(X)
    values = {}
    for col in X.columns:
        others = X.drop(columns=col)
        r2 = LinearRegression().fit(others, X[col]).score(others, X[col])
        values[col] = 1 / (1 - r2)
    return pd.Series(values)

rng = np.random.default_rng(2)
a = rng.normal(size=200)
df = pd.DataFrame({"a": a, "b": a + rng.normal(scale=0.1, size=200), "c": rng.normal(size=200)})
print("correlation a-b:", round(df.a.corr(df.b), 3))
vif(df).round(1)

## Leverage and influence

Not all points have the same power over the model. The **leverage** of an observation measures how far its
x value is from the center of the data: it is the diagonal element hᵢᵢ of the *hat matrix*
**H = A(AᵀA)⁻¹Aᵀ**, the one that turns y into ŷ. The mean leverage is p/n, where p is the number of columns of A.

A high-leverage point has the *potential* to move the line; it really does so only if it also has a large residual.
**Cook's distance** combines the two things:

```
Dᵢ = eᵢ² / (p · s²) · hᵢᵢ / (1 − hᵢᵢ)²         with  s² = SSR / (n − p)
```

A rule of thumb: Dᵢ > 4/n deserves a look, Dᵢ > 1 is almost always a problem.

In [ ]:
rng = np.random.default_rng(3)
x = np.r_[rng.uniform(0, 10, 30), 25.0]                 # the last point is far from the others
y = 2 + 1.5 * x + rng.normal(0, 1.5, len(x))
y[-1] -= 20                                             # and we move it a lot vertically

A = np.column_stack([np.ones_like(x), x])
H = A @ np.linalg.inv(A.T @ A) @ A.T                    # hat matrix: y_hat = H y
leverage = np.diag(H)

beta = np.linalg.lstsq(A, y, rcond=None)[0]
e = y - A @ beta
s2 = e @ e / (len(x) - 2)
cook = e ** 2 / (2 * s2) * leverage / (1 - leverage) ** 2

print("expected mean leverage (p/n):", round(2 / len(x), 3))
print("leverage of the far point   :", round(leverage[-1], 3))
print("maximum Cook's distance     :", round(cook.max(), 2), "on point", cook.argmax())

> **Common mistake.** Judging a linear model by R² and p-value without looking at the residuals: four datasets with the same line and the same R² (Anscombe's quartet) have completely different shapes, and a single point with a lot of leverage can move everything (exercises D2 and D3).

## Check your understanding

> *Click on a question to show its answer.*

<details><summary>Two features have correlation 0.98. Are the model predictions worse?</summary>

Almost never. Multicollinearity makes the individual coefficients unstable and hard to interpret, but their combination, that is the prediction, stays stable. The problem concerns whoever wants to explain the effect of a single variable, not whoever wants to predict.

</details>

<details><summary>A point has high leverage but a residual of almost zero. Is it a problem?</summary>

Usually not: it is a point far from the center that nevertheless sits on the line the others were already drawing. It increases the precision of the slope estimate. It becomes a problem only if it also has a large residual, and that is exactly what Cook's distance measures.

</details>